# Gold forecasting with LSTM

A compact LSTM uses the same data, output targets, folds and initial search budget as GRU.
Only the recurrent cell differs. Each fold starts with fresh weights and fresh fitted preprocessing.
The small initial architecture is deliberate because this dataset contains only a few thousand daily prices.

Run notebook 01 first. Review the shared audit and release assumptions before interpreting errors.

In [1]:
from pathlib import Path
import os
# Limit CPU threads before importing optional numerical/model libraries.
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from gold_common_utils import ROOT, load_config
# Load the shared experiment configuration.
cfg = load_config()
pd.set_option('display.max_columns', 20)
print('Project:', ROOT)
print('Target:', cfg['target_name'], '| horizons:', cfg['horizons'])

Project: /home/kosala/git-repos/exps/gold_forecasting
Target: Kitco Price (USD) | horizons: [1, 5, 20]


## 1. Load the shared panel and evaluation contract

In [2]:
from gold_common_utils import load_prepared, feature_columns, forecast_origins, metrics
from gold_model_utils import (DEFAULT_TRIALS, run_cross_validation, evaluate_holdout,
                              refit_and_forecast, fit_model, predict_model)
KIND = 'lstm'
# Reuse the processed data and folds created by notebook 01.
panel, features, folds = load_prepared()
display(folds)
print('Observed target:', panel.y.notna().sum())

,fold,cutoff,stop
0,cv_1,2175,2301
1,cv_2,2301,2427
2,cv_3,2427,2553
3,cv_4,2553,2679
4,cv_5,2679,2805
5,holdout,2805,3057


Observed target: 2646


## 2. Inspect input feature groups and candidate settings
`gold_only` establishes whether external factors improve an already useful gold-history model.
The initial search is intentionally small; expand it using development results only.
All candidate selection uses earlier inner validation. Read the adapters in `gold_model_utils.py`
for the explicit training and prediction logic.

In [3]:
# Show the exact candidates and the feature columns used per group.
trials = [dict(trial) for trial in DEFAULT_TRIALS[KIND]]
display(pd.DataFrame(trials))
for group in ['gold_only', 'external']:
    print(group, feature_columns(features, group, KIND))
# Change this local list to add candidates; keep comparable GRU/LSTM budgets.

,group,lookback,units
0,gold_only,20,32
1,external,20,32
2,external,60,32


gold_only ['gold_latest', 'gold_missing', 'gold_ret_5', 'gold_ret_20', 'gold_vol_20', 'year_sin', 'year_cos', 'weekday']
external ['gold_latest', 'gold_missing', 'gold_ret_5', 'gold_ret_20', 'gold_vol_20', 'year_sin', 'year_cos', 'weekday', 'dxy', 'fed', 'yield10', 'cpi_yoy', 'cpi_mom', 'gpr_ma7', 'gpr_acts_ma7', 'gpr_threats_ma7', 'brent_spot', 'wti', 'yield_spread', 'dxy_age_days', 'gpr_ma7_age_days', 'wti_missing']


## 3. Inspect sequence shapes and architecture

In [4]:
from gold_model_utils import sequence_rows, build_network
# Inspect the training window and network output shape before fitting.
example = trials[0]
cutoff = int(folds.iloc[0].cutoff)
rows = sequence_rows(panel, cutoff, example['lookback'], cfg['horizons'])
columns = feature_columns(features, example['group'], KIND)
print('Eligible windows:', len(rows))
print('Input shape: (batch, lookback, features) =', (None, example['lookback'], len(columns)))
print('Output shape:', (None, len(cfg['horizons'])))
network = build_network(KIND, example['lookback'], len(columns), len(cfg['horizons']), example['units'])
network.summary()

ValueError: output array is read-only

## 4. Run nested walk-forward development evaluation
This cell performs actual training and may take time. Each fold reports inner candidate scores,
selects a configuration, refits it, and predicts the outer block. Forecasting uses only information
available at each origin. Outer labels are used only for scoring.

For a quick installation check, use one fold and one candidate. Restore the full settings before
comparing models or making performance claims.

In [ ]:
# Full defaults: five outer folds and all candidates above.
cv_folds = folds
# Full training; results are checkpointed to artifacts/<kind>/ after each fold.
cv_predictions, cv_metrics, selections = run_cross_validation(
    KIND, panel, features, cv_folds, cfg, trials=trials)
display(cv_metrics)
display(selections)

## 5. Review errors by horizon, baseline skill, and prediction bias

In [ ]:
# Summarize outer error by horizon and check prediction bias per model.
display(cv_metrics.groupby('horizon')[['MAE', 'RMSE', 'MAPE', 'MASE', 'skill_vs_persistence']].mean())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cv_metrics.boxplot(column='MAE', by='horizon', ax=axes[0])
axes[0].set_title('Outer-fold MAE by horizon')
for horizon, group in cv_predictions.groupby('horizon'):
    observed = group.dropna(subset=['actual'])
    axes[1].scatter(observed.actual, observed.prediction - observed.actual, s=12, label=str(horizon))
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Observed gold price')
axes[1].set_ylabel('Forecast error')
axes[1].legend(title='Horizon')
plt.suptitle('')
plt.tight_layout()

## 6. Lock the final configuration and evaluate the untouched holdout
The default is off so you can review development results first. Set
`final_holdout_enabled` to true in `config.json` when ready. This is an experiment guard,
not a request for additional permission. The function chooses parameters using development
history, records them before scoring, and preserves existing holdout results.

In [ ]:
# Load the shared experiment configuration.
cfg = load_config()
# Holdout runs once, and parameters are locked before its results are scored.
if cfg['final_holdout_enabled']:
    holdout_predictions, holdout_metrics, locked_parameters = evaluate_holdout(
        KIND, panel, features, folds, cfg, trials=trials)
    display(holdout_metrics)
    display(locked_parameters)
else:
    print('Holdout not evaluated. Development predictions remain available for review.')

## 7. Refit the locked model and generate future forecasts
Run after the final holdout stage. The forecast starts from the latest observed gold quote.
Only the configured horizons are returned. Future actual external factors are not supplied.
Artifacts include the fitted model, input feature names, preprocessing where applicable,
locked configuration, predictions and metrics. No calibrated uncertainty intervals are claimed.

In [ ]:
locked = ROOT / 'artifacts' / KIND / 'locked_parameters.json'
if cfg['final_holdout_enabled'] and locked.exists():
    # Refit reuses the locked configuration from artifacts/<kind>/locked_parameters.json.
    future_forecast = refit_and_forecast(KIND, panel, features, cfg)
    display(future_forecast)
else:
    print('Final refit waits for the parameter-lock stage.')